In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# อินพุตและเอาต์พุตของ Estimator

<Accordion>
<AccordionItem title="เวอร์ชันแพ็กเกจ">

โค้ดในหน้านี้พัฒนาโดยใช้ requirements ต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
หน้านี้ให้ภาพรวมของ inputs และ outputs ของ Qiskit Runtime Estimator primitive ซึ่งรัน workloads บน IBM Quantum&reg; compute resources Estimator ให้คุณกำหนด vectorized workloads ได้อย่างมีประสิทธิภาพโดยใช้โครงสร้างข้อมูลที่เรียกว่า [**Primitive Unified Bloc (PUB)**](/guides/primitive-input-output#pubs) ซึ่งใช้เป็น inputs ของเมธอด [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) สำหรับ Estimator primitive ซึ่งรัน workload ที่กำหนดเป็น job จากนั้น หลังจาก job เสร็จสิ้น ผลลัพธ์จะถูกส่งคืนในรูปแบบที่ขึ้นอยู่กับทั้ง PUBs ที่ใช้และ runtime options ที่ระบุจาก primitive
## Inputs

แต่ละ PUB อยู่ในรูปแบบนี้:

(`<single circuit>`, `<one or more observables>`, `<optional one or more parameter values>`, `<optional precision>`),

`parameter values` ทางเลือกสามารถเป็น list หรือ parameter เดียว Elements จาก observables และ parameter values จะถูกรวมกันโดยทำตาม NumPy broadcasting rules ตามที่อธิบายในหัวข้อ [Primitive inputs and outputs](primitive-input-output#broadcasting-rules) และหนึ่ง expectation value estimate จะถูกส่งคืนสำหรับแต่ละ element ของ broadcasted shape

> **Note:** ถ้า input มีการวัด จะถูก ignore

สำหรับ Estimator primitive PUB สามารถมีได้สูงสุดสี่ค่า:
- `QuantumCircuit` เดียว ซึ่งอาจมี object [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter) หนึ่งรายการหรือมากกว่า
- List ของ observables หนึ่งรายการหรือมากกว่า ซึ่งระบุ expectation values ที่จะประมาณ จัดเป็น array (ตัวอย่างเช่น observable เดียวแสดงเป็น 0-d array, list ของ observables เป็น 1-d array และอื่นๆ) ข้อมูลสามารถอยู่ในรูปแบบ `ObservablesArrayLike` ใดก็ได้ เช่น `Pauli`, `SparsePauliOp`, `PauliList` หรือ `str`
   > **Note:** - Commuting observables **ใน PUB เดียวกัน** จะถูกจัดกลุ่มโดยใช้ [วิธีนี้](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting)
>    - Commuting observables ใน PUBs ที่แตกต่างกัน แม้ว่าจะมี circuit เดียวกัน จะไม่ถูกประมาณโดยใช้การวัดเดียวกัน แต่ละ PUB แสดง basis ที่แตกต่างกันสำหรับการวัด ดังนั้น จำเป็นต้องวัดแยกกันสำหรับแต่ละ PUB
>    - เพื่อให้แน่ใจว่า commuting observables ถูกประมาณโดยใช้การวัดเดียวกัน จัดกลุ่มไว้ใน PUB เดียวกัน
- คอลเลกชันของค่า parameter เพื่อ bind circuit กับสิ่งนี้ สามารถระบุเป็น single array-like object ที่ last index อยู่เหนือ circuit `Parameter` objects หรือละเว้น (หรือเทียบเท่ากัน ตั้งเป็น `None`) ถ้า circuit ไม่มี `Parameter` objects
- (ทางเลือก) Target precision สำหรับ expectation values ที่จะประมาณ
---

โค้ดต่อไปนี้แสดงตัวอย่างชุด vectorized inputs ให้กับ `Estimator` primitive และรันบน IBM&reg; backend เป็น `RuntimeJobV2` object เดียว

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
)

import numpy as np

# Instantiate runtime service and get
# the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 100),
        np.linspace(-4 * np.pi, 4 * np.pi, 100),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator_pub = (transpiled_circuit, observables, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
estimator = Estimator(mode=backend)
job = estimator.run([estimator_pub])
result = job.result()

## Outputs

หลังจากส่ง PUBs หนึ่งรายการหรือมากกว่าไปยัง QPU เพื่อรัน และ job เสร็จสำเร็จ ข้อมูลจะถูกส่งคืนเป็น container object [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) ที่เข้าถึงได้โดยการเรียกเมธอด `RuntimeJobV2.result()`

`PrimitiveResult` มี iterable list ของ object [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) ที่มีผลการรันสำหรับแต่ละ PUB

แต่ละ element ของ list นี้สอดคล้องกับแต่ละ PUB ที่ส่งไปยังเมธอด `run()` ของ primitive (ตัวอย่างเช่น job ที่ส่งกับ 20 PUBs จะส่งคืน `PrimitiveResult` object ที่มี list ของ `PubResult` objects 20 รายการ หนึ่งสอดคล้องกับแต่ละ PUB)

แต่ละ `PubResult` สำหรับ Estimator primitive มีอย่างน้อย array ของ expectation values (`PubResult.data.evs`) และ standard deviations ที่เกี่ยวข้อง (ไม่ว่าจะเป็น `PubResult.data.stds` หรือ `PubResult.data.ensemble_standard_error` ขึ้นอยู่กับ `resilience_level` ที่ใช้) แต่อาจมีข้อมูลเพิ่มเติมขึ้นอยู่กับ error mitigation options ที่ระบุ

แต่ละ `PubResult` object มีทั้ง `data` และ `metadata` attribute
    - Attribute `data` เป็น [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) ที่กำหนดเองซึ่งมีค่าการวัดจริง standard deviations และอื่นๆ
    - `DataBin` มี attributes ต่างๆ ขึ้นอยู่กับ shape หรือโครงสร้างของ PUB ที่เกี่ยวข้อง รวมถึง error mitigation options ที่ระบุโดย primitive ที่ใช้ส่ง job (เช่น [ZNE](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) หรือ [PEC](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec))
    - Attribute `metadata` มีข้อมูลเกี่ยวกับ runtime และ error mitigation options ที่ใช้ (อธิบายในส่วน [Result metadata](#result-metadata) ของหน้านี้)

ต่อไปนี้คือโครงร่างภาพของโครงสร้างข้อมูล `PrimitiveResult` สำหรับ Estimator output:

    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```

พูดง่ายๆ คือ job เดียวส่งคืน `PrimitiveResult` object และมี list ของ `PubResult` objects หนึ่งรายการหรือมากกว่า `PubResult` objects เหล่านี้เก็บข้อมูลการวัดสำหรับแต่ละ PUB ที่ส่งไปยัง job

code snippet ด้านล่างอธิบายรูปแบบ `PrimitiveResult` (และ `PubResult` ที่เกี่ยวข้อง) สำหรับ job ที่สร้างด้านบน

In [2]:
print(
    f"The result of the submitted job had {len(result)} PUB and has a value:\n {result}\n"
)
print(
    f"The associated PubResult of this job has the following data bins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the\n\
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). \n"
)
with np.printoptions(threshold=200):
    print(
        f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
    )

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=(3, 100), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(3, 100), dtype=float64>), shape=(3, 100)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=

#### How the Estimator primitive calculates error

In addition to the estimate of the mean of the observables passed in the input PUBs (the `evs` field of the `DataBin`), Estimator also attempts to deliver an estimate of the error associated with those expectation values. All Estimator queries will populate the `stds` field with a quantity like the standard error of the mean for each expectation value, but some error mitigation options produce additional information, such as `ensemble_standard_error`.

Consider a single observable $\mathcal{O}$. In the absence of [ZNE](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne), you can think of each shot of the Estimator execution as providing a point estimate of the expectation value $\langle \mathcal{O} \rangle$. If the pointwise estimates are in a vector `Os`, then the value returned in `ensemble_standard_error` is equivalent to the following (in which $\sigma_{\mathcal{O}}$ is the [standard deviation of the expectation value](/docs/api/qiskit/qiskit.primitives.BackendEstimatorV2) estimate and $N_{shots}$ is the number of shots):

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{shots}} },$$

which treats all shots as part of a single ensemble. If you requested gate [twirling](/docs/guides/error-mitigation-and-suppression-techniques#pauli-twirling) (`twirling.enable_gates = True`), you can sort the pointwise estimates of $\langle \mathcal{O} \rangle$ into sets that share a common twirl. Call these sets of estimates `O_twirls`, and there are `num_randomizations` (number of twirls) of them. Then `stds` is the standard error of the mean of `O_twirls`, as in

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{twirls}} },$$

where $\sigma_{\mathcal{O}}$ is the standard deviation of `O_twirls` and $N_{twirls}$ is the number of twirls. When you do not enable twirling, `stds` and `ensemble_standard_error` are equal.

If you enable ZNE, then the `stds` described above become weights in a non-linear regression to an extrapolator model. What finally gets returned in the `stds` in this case is the uncertainty of the fit model evaluated at a noise factor of zero. When there is a poor fit, or large uncertainty in the fit, the reported `stds` can become very large. When ZNE is enabled, `pub_result.data.evs_noise_factors` and `pub_result.data.stds_noise_factors` are also populated, so that you can do your own extrapolation.

## Result metadata

In addition to the execution results, both the `PrimitiveResult` and `PubResult` objects contain a metadata attribute about the job that was submitted. The metadata containing information for all submitted PUBs (such as the various [runtime options](/docs/api/qiskit-ibm-runtime/options) available) can be found in the `PrimitiveResult.metatada`, while the metadata specific to each PUB is found in `PubResult.metadata`.

<Admonition type="note">
In the metadata field, primitive implementations can return any information about execution that is relevant to them, and there are no key-value pairs that are guaranteed by the base primitive. Thus, the returned metadata might be different in different primitive implementations.
</Admonition>

In [3]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'dynamical_decoupling' : {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'},
'twirling' : {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'},
'resilience' : {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False},
'version' : 2,

The metadata of the PubResult result is:
'shots' : 4096,
'target_precision' : 0.015625,
'circuit_metadata' : {},
'resilience' : {},
'num_randomizations' : 32,
